# 🌾 Módulo 1: Predicción Supervisada de Rendimiento Agrícola ante El Niño
## Proyecto: AgroSmart Perú — Universidad / Software Inteligente

**Objetivo:** Predecir el rendimiento agrícola ($t/ha$) de cultivos clave en Piura (Limón y Arroz) a partir de variables climáticas y oceanográficas (Índice ICEN, anomalía TSM Niño 1+2, lluvias y temperatura).  
**Datos Reales:** BCRP (Producción mensual 2007–2026), NOAA y SENAMHI.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
os.makedirs('../models', exist_ok=True)
print("Librerías importadas correctamente.")


### 1. Carga y Exploración de Datos Reales


In [ ]:
df = pd.read_csv('../data/tabla_maestra_agro_nino.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df[['cultivo', 'icen', 'anomalia_tsm', 'lluvia_mensual_mm', 'rendimiento_tha']].head(8)


In [ ]:
# Matriz de correlación para el cultivo de Limón
df_limon = df[df['cultivo'] == 'Limon'].copy()
corr = df_limon[['icen', 'anomalia_tsm', 'lluvia_mensual_mm', 'temp_max_prom', 'rendimiento_tha']].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='RdYlGn_r', fmt='.2f')
plt.title('Correlación: Factores El Niño vs Rendimiento de Limón (Piura)')
plt.show()


### 2. Preparación de Datos y Partición Train/Test (80/20)


In [ ]:
features = ['icen', 'anomalia_tsm', 'lluvia_mensual_mm', 'temp_max_prom', 'mes']
target = 'rendimiento_tha'

X = df_limon[features]
y = df_limon[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Muestras de Entrenamiento: {X_train.shape[0]} | Muestras de Prueba: {X_test.shape[0]}")


### 3. Entrenamiento y Comparación de Modelos
Comparamos 4 algoritmos:
1. Regresión Lineal (Baseline)
2. Regresión Ridge
3. Gradient Boosting Regressor
4. **Random Forest Regressor** (Modelo Propuesto)


In [ ]:
modelos = {
    'Regresión Lineal': LinearRegression(),
    'Ridge Regressor': Ridge(alpha=1.0),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
}

resultados = []
for nombre, m in modelos.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    resultados.append({'Modelo': nombre, 'R² Score': round(r2, 4), 'MAE (t/ha)': round(mae, 3), 'RMSE (t/ha)': round(rmse, 3)})

df_eval = pd.DataFrame(resultados).sort_values('R² Score', ascending=False)
print("📊 Tabla de Evaluación de Rendimiento:")
df_eval


### 4. Gráfica de Evaluación: Valores Reales vs Predichos e Importancia de Variables


In [ ]:
rf_final = modelos['Random Forest']
y_pred_rf = rf_final.predict(X_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Real vs Predicho
ax1.scatter(y_test, y_pred_rf, color='#2e7d32', edgecolors='k', alpha=0.7)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_xlabel('Rendimiento Real (t/ha)')
ax1.set_ylabel('Rendimiento Predicho (t/ha)')
ax1.set_title(f'Random Forest: Ajuste Real vs Predicho (R² = {r2_score(y_test, y_pred_rf):.3f})')

# Importancia de Variables
imp = pd.Series(rf_final.feature_importances_, index=features).sort_values()
imp.plot(kind='barh', color='#388e3c', ax=ax2)
ax2.set_xlabel('Peso Relativo en la Decisión')
ax2.set_title('Variables más Determinantes en la Merma de Rendimiento')

plt.tight_layout()
plt.show()


### 5. Exportación del Modelo Serializado (.pkl)


In [ ]:
joblib.dump(rf_final, '../models/rf_yield_limon.pkl')
print("✅ Modelo exportado exitosamente a '../models/rf_yield_limon.pkl'")

# Entrenamiento y exportación para Arroz
df_arroz = df[df['cultivo'] == 'Arroz'].copy()
X_a, y_a = df_arroz[features], df_arroz[target]
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_a, y_a, test_size=0.2, random_state=42)
rf_arroz = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf_arroz.fit(X_train_a, y_train_a)
joblib.dump(rf_arroz, '../models/rf_yield_arroz.pkl')
print(f"✅ Modelo de Arroz exportado (R² = {r2_score(y_test_a, rf_arroz.predict(X_test_a)):.3f})")
